In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, GBTRegressor, LinearRegressionModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, CrossValidatorModel
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
import os
import pickle
import subprocess
import json
import builtins
warnings.filterwarnings('ignore')

In [0]:
%pip install kaggle

In [0]:
os.environ['KAGGLE_USERNAME'] = 'raflikp' # Pakai username Kaggle sendiri
os.environ['KAGGLE_KEY'] = '76aec97260e5b2ea3e3d3b7269fd31c5'

In [0]:
download_path = "/Volumes/workspace/default/nyc_taxi_volume"
os.chdir(download_path)

# Download train.csv file
subprocess.run([
    "kaggle", "competitions", "download",
    "-c", "new-york-city-taxi-fare-prediction",
    "-f", "train.csv"
], check=True)

print(f"Download selesai! File tersimpan di {download_path}")

In [0]:
VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"

for f in os.listdir(VOLUME_PATH):
    size_mb = os.path.getsize(f"{VOLUME_PATH}/{f}") / (1024**2)
    print(f"  {f} — {size_mb:.1f} MB")

In [0]:
import zipfile

zip_path   = f"{VOLUME_PATH}/train.csv.zip"
extract_to = VOLUME_PATH

print("Mulai unzip...")
with zipfile.ZipFile(zip_path, 'r') as z:
    members = z.namelist()
    print(f"File di dalam zip: {members}")
    z.extractall(extract_to)
    print(f"Extracted ke: {extract_to}")

print("\nIsi Volume setelah unzip:")
for f in os.listdir(VOLUME_PATH):
    size_mb = os.path.getsize(f"{VOLUME_PATH}/{f}") / (1024**2)
    print(f"  {f} — {size_mb:.1f} MB")

In [0]:
spark = SparkSession.builder \
    .appName("NYC_Taxi_Full_Pipeline") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

schema = StructType([
    StructField("key",               StringType(),  True),
    StructField("fare_amount",       DoubleType(),  True),
    StructField("pickup_datetime",   StringType(),  True),
    StructField("pickup_longitude",  DoubleType(),  True),
    StructField("pickup_latitude",   DoubleType(),  True),
    StructField("dropoff_longitude", DoubleType(),  True),
    StructField("dropoff_latitude",  DoubleType(),  True),
    StructField("passenger_count",   IntegerType(), True),
])

def checkpoint(df, name):
    path = f"{VOLUME_PATH}/checkpoints/{name}"
    df.write.mode("overwrite").parquet(path)
    df_loaded = spark.read.parquet(path)
    print(f"Checkpoint '{name}' disimpan & dimuat ulang.")
    return df_loaded

print("Loading train.csv...")
df_raw = spark.read.csv(
    f"{VOLUME_PATH}/train.csv",
    header=True,
    schema=schema
)
df_raw.show(5)

In [0]:
print("Jumlah data:", df_raw.count())
print("Jumlah kolom:", len(df_raw.columns))
df_raw.printSchema()
df_raw.explain()

In [0]:
df_raw.describe().show()

In [0]:
df_raw.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
]).show()

In [0]:
df_clean = df_raw.dropna(subset=[
    "dropoff_longitude",
    "dropoff_latitude"
])

In [0]:
total_rows = df_raw.count()
unique_keys = df_raw.select("key").distinct().count()

print("Total:", total_rows)
print("Duplikat:", total_rows - unique_keys)

In [0]:
duplicate_keys = df_raw.groupBy("key").count().filter("count > 1")

df_only_duplicates = df_raw.join(duplicate_keys, on="key", how="inner")

df_only_duplicates.orderBy("key").show(truncate=False)

In [0]:
total_rows = df_raw.count()
unique_rows = df_raw.dropDuplicates().count()

print("Total:", total_rows)
print("Duplikat:", total_rows - unique_rows)

In [0]:
df_dup_keys = df_raw.groupBy("key") \
                   .count() \
                   .filter("count > 1") \
                   .select("key")

df_full_dup = df_raw.join(df_dup_keys, on="key", how="inner")

df_full_dup.show(truncate=False)

In [0]:
df_base = (
    df_raw
    .drop("key")
    .dropna()
)

df_base.show(5)

In [0]:
df_time = (
    df_base
    .withColumn("pickup_ts", F.to_timestamp("pickup_datetime", "yyyy-MM-dd HH:mm:ss 'UTC'"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn("pickup_dayofweek", F.dayofweek("pickup_ts"))
    .withColumn("pickup_month", F.month("pickup_ts"))
    .withColumn("pickup_year", F.year("pickup_ts"))
)

df_time.show(5)

In [0]:
R = 6371

def add_haversine(df):   
    df_with_distance = df.withColumn(
        "distance_km",
        2 * R * atan2(
            sqrt(
                sin((radians(col("dropoff_latitude") - col("pickup_latitude")) / 2))**2 +
                cos(radians(col("pickup_latitude"))) *
                cos(radians(col("dropoff_latitude"))) *
                sin((radians(col("dropoff_longitude") - col("pickup_longitude")) / 2))**2
            ),
            sqrt(
                1 - (
                    sin((radians(col("dropoff_latitude") - col("pickup_latitude")) / 2))**2 +
                    cos(radians(col("pickup_latitude"))) *
                    cos(radians(col("dropoff_latitude"))) *
                    sin((radians(col("dropoff_longitude") - col("pickup_longitude")) / 2))**2
                )
            )
        )
    )
    return df_with_distance

df_distance = add_haversine(df_time)
df_distance.select("distance_km").show(5)

In [0]:
df_clean = (
    df_distance
    .filter((col("distance_km") > 0) & (col("distance_km") < 100))
    .filter((col("fare_amount") > 0) & (col("fare_amount") < 500))
    .filter((col("passenger_count").between(1, 6)))
    .filter(col("pickup_longitude").between(-74.2591, -73.7004))
    .filter(col("pickup_latitude").between(40.4774, 40.9176))
    .filter(col("dropoff_longitude").between(-74.2591, -73.7004))
    .filter(col("dropoff_latitude").between(40.4774, 40.9176))
)

df_clean.show(5)

In [0]:
df_final = (
    df_clean
    .withColumn(
        "day_type",
        F.when(F.col("pickup_dayofweek").isin(1,7), "weekend")
         .otherwise("weekday")
    )
    .withColumn(
        "rush_hour",
        F.when((F.col("pickup_hour").between(7,9)) | (F.col("pickup_hour").between(16,19)), 1)
         .otherwise(0)
    )
    .withColumn(
        "time_of_day",
        F.when(col("pickup_hour") < 6, "night")
         .when(col("pickup_hour") < 12, "morning")
         .when(col("pickup_hour") < 18, "afternoon")
         .otherwise("evening")
    )
    .withColumn(
        "rush_hour",
        F.when(
            (col("pickup_hour").between(7,9)) | 
            (col("pickup_hour").between(16,19)), 1
        ).otherwise(0)
    )
    .withColumn(
        "trip_type",
        F.when(col("distance_km") < 2, "short")
         .when(col("distance_km") < 10, "medium")
         .otherwise("long")
    )
)

df_final.show(5, truncate = False)

In [0]:
df_final.select(
    sum((col("fare_amount") <= 0).cast("int")).alias("fare_anomali"),

    sum(((col("passenger_count") <= 0) | (col("passenger_count") > 6)).cast("int"))
        .alias("passenger_anomali"),

    sum(((col("distance_km") == 0) | (col("distance_km") > 100)).cast("int"))
        .alias("distance_anomali"),

    sum(((col("pickup_longitude") < -74.2591) | (col("pickup_longitude") > -73.7004)).cast("int"))
        .alias("pickup_longitude_anomali"),

    sum(((col("dropoff_longitude") < -74.2591) | (col("dropoff_longitude") > -73.7004)).cast("int"))
        .alias("dropoff_longitude_anomali"),

    sum(((col("pickup_latitude") < 40.4774) | (col("pickup_latitude") > 40.9176)).cast("int"))
        .alias("pickup_latitude_anomali"),

    sum(((col("dropoff_latitude") < 40.4774) | (col("dropoff_latitude") > 40.9176)).cast("int"))
        .alias("dropoff_latitude_anomali")

).show()

In [0]:
df_final.groupBy("day_type") \
   .agg(
       F.min("fare_amount").alias("min"),
       F.max("fare_amount").alias("max"),
       F.avg("fare_amount").alias("avg")
   ) \
   .show()

In [0]:
df_clean.groupBy("passenger_count") \
        .count() \
        .orderBy(col("count").desc()) \
        .show()

In [0]:
df_clean.select("fare_amount", "distance_km").show(10)

In [0]:
cols = [
    "fare_amount",
    "distance_km",
    "passenger_count",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "pickup_year"
]

In [0]:
df_sample = df_final.select(cols).sample(0.1).toPandas()

In [0]:
corr = df_sample.corr()

In [0]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    linecolor="white"
)

plt.title("Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df_sample,
    x="distance_km",
    y="fare_amount",
    alpha=0.3
)
plt.title("Fare vs Distance")
plt.show()

In [0]:
df_distance.show(10, truncate = False)

## PREPROCESSING

In [0]:
print("=" * 60)
print("PREPROCESSING")
print("=" * 60)

df_parsed = df_raw \
    .withColumn(
        "pickup_dt",
        to_timestamp(
            regexp_replace(col("pickup_datetime"), " UTC$", ""),
            "yyyy-MM-dd HH:mm:ss"
        )
    ) \
    .withColumn("hour",        hour("pickup_dt")) \
    .withColumn("day_of_week", dayofweek("pickup_dt")) \
    .withColumn("month",       month("pickup_dt")) \
    .withColumn("year",        year("pickup_dt")) \
    .withColumn("is_weekend",
        when(dayofweek("pickup_dt").isin([1, 7]), 1).otherwise(0)
    )

df_filtered = df_parsed.filter(
    col("fare_amount").between(2.5, 500)             &
    col("pickup_longitude").between(-74.05, -73.75)  &
    col("pickup_latitude").between(40.60, 40.90)     &
    col("dropoff_longitude").between(-74.05, -73.75) &
    col("dropoff_latitude").between(40.60, 40.90)    &
    col("passenger_count").between(1, 6)             &
    col("pickup_dt").isNotNull()
)

# Checkpoint setelah filtering
df_filtered = checkpoint(df_filtered, "filtered")

count_filtered = df_filtered.count()
print(f"Baris sebelum filter : {total_rows:,}")
print(f"Baris setelah filter : {count_filtered:,}")
print(f"Baris dihapus        : {total_rows - count_filtered:,}")

In [0]:
df_feat = add_haversine(df_filtered)

df_feat = df_feat \
    .withColumn("abs_lat_diff", abs(col("dropoff_latitude")  - col("pickup_latitude"))) \
    .withColumn("abs_lon_diff", abs(col("dropoff_longitude") - col("pickup_longitude"))) \
    .withColumn("is_rush_hour",
        when(
            ((col("hour") >= 7)  & (col("hour") <= 9)) |
            ((col("hour") >= 17) & (col("hour") <= 19)), 1
        ).otherwise(0)
    ) \
    .withColumn("is_night",
        when((col("hour") >= 22) | (col("hour") <= 5), 1).otherwise(0)
    )

FEATURE_COLS = [
    "distance_km", "abs_lat_diff", "abs_lon_diff",
    "hour", "day_of_week", "month", "year",
    "passenger_count", "is_weekend", "is_rush_hour", "is_night"
]
TARGET_COL = "fare_amount"

df_model = df_feat.select(FEATURE_COLS + [TARGET_COL]).dropna()

df_model = checkpoint(df_model, "model_ready")

print(f"Data siap model: {df_model.count():,} baris | {len(FEATURE_COLS)} fitur")
df_model.show(5, truncate = False)

In [0]:
assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features",      
    handleInvalid="skip"
)

df_scaled = assembler.transform(df_model).select("features", TARGET_COL)

df_scaled = checkpoint(df_scaled, "scaled")

train_df, val_df, test_df = df_scaled.randomSplit([0.70, 0.15, 0.15], seed=42)

train_df = checkpoint(train_df, "split_train")
val_df   = checkpoint(val_df,   "split_val")
test_df  = checkpoint(test_df,  "split_test")

print(f"Train : {train_df.count():,}")
print(f"Val   : {val_df.count():,}")
print(f"Test  : {test_df.count():,}")

In [0]:
# Loading Checkpoint incase session wiped out

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi_volume"
FEATURE_COLS = [
    "distance_km", "abs_lat_diff", "abs_lon_diff",
    "hour", "day_of_week", "month", "year",
    "passenger_count", "is_weekend", "is_rush_hour", "is_night"
]
TARGET_COL = "fare_amount"

train_df = spark.read.parquet(f"{VOLUME_PATH}/checkpoints/{"split_train"}")
val_df = spark.read.parquet(f"{VOLUME_PATH}/checkpoints/{"split_val"}")
test_df = spark.read.parquet(f"{VOLUME_PATH}/checkpoints/{"split_test"}")
df_model = spark.read.parquet(f"{VOLUME_PATH}/checkpoints/{"model_ready"}")

print(f"Train : {train_df.count():,}")
train_df.show(5, truncate = False)
print(f"Val   : {val_df.count():,}")
val_df.show(5, truncate = False)
print(f"Test  : {test_df.count():,}")
test_df.show(5, truncate = False)
print(f"Model  : {df_model.count():,}")
df_model.show(5, truncate = False)


In [0]:
# Sampling agar tidak berat saat testing (Optional)

train_df_sample = train_df.sample(withReplacement=False, fraction=0.1, seed=42)
val_df_sample = val_df.sample(withReplacement=False, fraction=0.1, seed=42)
test_df_sample = test_df.sample(withReplacement=False, fraction=0.1, seed=42)
                                
print(f"Training sample: {train_df_sample.count():,}")
print(f"Validation sample: {val_df_sample.count():,}")
print(f"Test sample: {test_df_sample.count():,}")

In [0]:
# Metrik Evaluasi (RMSE, MAE, R²)

def evaluate(preds, model_name):
    rmse = eval_rmse.evaluate(preds)
    mae  = eval_mae.evaluate(preds)
    r2   = eval_r2.evaluate(preds)
    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R²   : {r2:.4f}")
    return {"model": model_name, "RMSE": rmse, "MAE": mae, "R2": r2}

## Training Linear Regression

In [0]:
TARGET_COL = "fare_amount"
CKPT       = f"{VOLUME_PATH}/checkpoints"
LR_MODEL_PATH  = f"{VOLUME_PATH}/checkpoints/lr_model"
LR_RESULT_PATH = f"{CKPT}/lr_result.json"

eval_rmse = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")

print("Training Linear Regression...")

lr = LinearRegression(
    featuresCol="features",
    labelCol=TARGET_COL,
    maxIter=100,         
    regParam=0.01,        
    elasticNetParam=0.0,  
    standardization=True  
)

lr_model  = lr.fit(train_df)
lr_preds  = lr_model.transform(test_df)
lr_result = evaluate(lr_preds, "Linear Regression")

lr_model.write().overwrite().save(LR_MODEL_PATH)
print(f"\nModel disimpan  : {LR_MODEL_PATH}")

with open(LR_RESULT_PATH, "w") as f:
    json.dump(lr_result, f, indent=2)
print(f"Result disimpan : {LR_RESULT_PATH}")


print(f"\n  Training summary:")
print(f"  Iterasi           : {lr_model.summary.totalIterations}")
print(f"  Loss awal         : {lr_model.summary.objectiveHistory[0]:.6f}")
print(f"  Loss akhir        : {lr_model.summary.objectiveHistory[-1]:.6f}")

print(f"\n  Koefisien per fitur:")
for name, coef in zip(FEATURE_COLS, lr_model.coefficients):
    print(f"    {name:<20}: {coef:+.6f}")
print(f"  Intercept         : {lr_model.intercept:+.6f}")

In [0]:
print("Training Gradient Boosted Trees...")
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8,
    minInstancesPerNode=5,
    seed=42
)
gbt_model  = gbt.fit(train_df)
gbt_preds  = gbt_model.transform(test_df)
gbt_result = evaluate(gbt_preds, "Gradient Boosted Trees (50 iter, depth 5)")

print("\n  Feature Importances (GBT):")
gbt_importances = sorted(
    zip(FEATURE_COLS, gbt_model.featureImportances),
    key=lambda x: x[1], reverse=True
)
for name, imp in gbt_importances:
    bar = "█" * int(imp * 60)
    print(f"    {name:<20}: {bar} {imp:.4f}")

from pyspark.ml.regression import GBTRegressionModel
import json

GBT_MODEL_PATH = f"{VOLUME_PATH}/checkpoints/gbt_model"
GBT_PREDS_PATH = f"{VOLUME_PATH}/checkpoints/gbt_predictions"
GBT_META_PATH  = f"{VOLUME_PATH}/checkpoints/gbt_model/metadata.json"

gbt_model.write().overwrite().save(GBT_MODEL_PATH)
print(f"\nModel disimpan     : {GBT_MODEL_PATH}")

gbt_preds.select(TARGET_COL, "prediction") \
    .write.mode("overwrite") \
    .parquet(GBT_PREDS_PATH)
print(f"Predictions disimpan: {GBT_PREDS_PATH}")

gbt_meta = {
    "model_name"  : "Gradient Boosted Trees",
    "params": {
        "maxIter"             : gbt_model.getMaxIter(),
        "maxDepth"            : gbt_model.getMaxDepth(),
        "stepSize"            : gbt_model.getStepSize(),
        "subsamplingRate"     : gbt_model.getSubsamplingRate(),
        "minInstancesPerNode" : gbt_model.getMinInstancesPerNode(),
    },
    "metrics": {
        "RMSE" : gbt_result["RMSE"],
        "MAE"  : gbt_result["MAE"],
        "R2"   : gbt_result["R2"],
    },
    "feature_importances": {
        name: float(imp) for name, imp in gbt_importances
    }
}
with open(GBT_META_PATH, "w") as f:
    json.dump(gbt_meta, f, indent=2)
print(f"Metadata disimpan  : {GBT_META_PATH}")

# 4. Verifikasi semua tersimpan
print("\n" + "=" * 50)
print("CHECKPOINT SUMMARY — GBT")
print("=" * 50)
for label, path in [
    ("Model",       GBT_MODEL_PATH),
    ("Predictions", GBT_PREDS_PATH),
    ("Metadata",    GBT_META_PATH),
]:
    status = "✓" if os.path.exists(path) else "✗"
    print(f"  {status} {label:<15}: {path}")

In [0]:
from pyspark.ml.regression import GBTRegressionModel
import json

CKPT       = f"{VOLUME_PATH}/checkpoints"
LR_MODEL_PATH  = f"{VOLUME_PATH}/checkpoints/lr_model"
LR_RESULT_PATH = f"{CKPT}/lr_result.json"

gbt_model = GBTRegressionModel.load(f"{VOLUME_PATH}/checkpoints/gbt_model")
lr_model   = LinearRegressionModel.load(LR_MODEL_PATH)
with open(LR_RESULT_PATH, "r") as f:
    lr_result = json.load(f)

gbt_preds = spark.read.parquet(f"{VOLUME_PATH}/checkpoints/gbt_predictions")
lr_preds  = lr_model.transform(test_df)

with open(f"{VOLUME_PATH}/checkpoints/gbt_model/metadata.json") as f:
    gbt_meta = json.load(f)

with open(LR_RESULT_PATH, "r") as f:
    lr_meta = json.load(f)

gbt_result = evaluate(gbt_preds, "Gradient Boosted Trees (50 iter, depth 5)")
lr_result = evaluate(lr_preds, "Linear Regression")

In [0]:
all_results = [lr_result, gbt_result]
results_df  = pd.DataFrame(all_results).set_index("model")

print("\n" + "=" * 55)
print("PERBANDINGAN SEMUA MODEL")
print("=" * 55)
print(results_df.to_string())

best_model_name = results_df["RMSE"].idxmin()
print(f"\nModel terbaik (RMSE terendah): {best_model_name}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics   = ["RMSE", "MAE", "R2"]
colors    = ["#E86452", "#F6BD16", "#61DDAA"]

for ax, metric, color in zip(axes, metrics, colors):
    vals  = results_df[metric]
    bars  = ax.bar(range(len(vals)), vals, color=color, edgecolor='white', alpha=0.85)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(
        ["Linear\nRegression", "GBT"],
        fontsize=9
    )
    ax.set_title(f"Perbandingan {metric}", fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.002,
                f"{v:.4f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f"{VOLUME_PATH}/eval_model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [0]:
best_preds = gbt_preds

pdf_eval = best_preds.select(TARGET_COL, "prediction") \
    .sample(fraction=0.005, seed=42) \
    .toPandas()

pdf_eval.columns = ["actual", "predicted"]
pdf_eval["residual"] = pdf_eval["actual"] - pdf_eval["predicted"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

import builtins

max_val = builtins.min(float(pdf_eval["actual"].max()), 150.0)
axes[0].scatter(pdf_eval["actual"], pdf_eval["predicted"],
                alpha=0.25, s=8, color="#5B8FF9")
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Perfect prediction")
axes[0].set_xlim(0, max_val)
axes[0].set_ylim(0, max_val)
axes[0].set_title("GBT — Actual vs Predicted", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Actual Fare (USD)")
axes[0].set_ylabel("Predicted Fare (USD)")
axes[0].legend()

axes[1].scatter(pdf_eval["predicted"], pdf_eval["residual"],
                alpha=0.25, s=8, color="#E86452")
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_title("Residual Plot", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Predicted Fare (USD)")
axes[1].set_ylabel("Residual (Actual − Predicted)")

plt.tight_layout()
plt.savefig(f"{VOLUME_PATH}/eval_actual_vs_predicted.png", dpi=150, bbox_inches='tight')
plt.show()

In [0]:
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/nyc_taxi_volume/sparkml_tmp"

CKPT = f"{VOLUME_PATH}/checkpoints"

print("=" * 60)
print("HYPERPARAMETER TUNING — GBT dengan CrossValidator")
print("=" * 60)

print("Cek kolom df_model:")
print(df_model.columns)

train_cv, _ = df_model.randomSplit([0.1, 0.9], seed=42)
train_cv     = checkpoint(train_cv, "cv_sample_v2")
print(f"Data CV    : {train_cv.count():,} baris")
print(f"Data train : {train_df.count():,} baris (referensi GBT Default)")

ASSEMBLER_PATH = f"{CKPT}/assembler_t"
if os.path.exists(ASSEMBLER_PATH):
    print("Memuat assembler_t dari checkpoint...")
    assembler_t = VectorAssembler.load(ASSEMBLER_PATH)
else:
    print("Membuat assembler_t baru...")
    assembler_t = VectorAssembler(
        inputCols=FEATURE_COLS,
        outputCol="features",
        handleInvalid="skip"
    )
    assembler_t.save(ASSEMBLER_PATH)
    print(f"assembler_t disimpan di: {ASSEMBLER_PATH}")

GBT_T_PATH = f"{CKPT}/gbt_t"
if os.path.exists(GBT_T_PATH):
    print("Memuat gbt_t dari checkpoint...")
    gbt_t = GBTRegressor.load(GBT_T_PATH)
else:
    print("Membuat gbt_t baru...")
    gbt_t = GBTRegressor(
        featuresCol="features",
        labelCol=TARGET_COL,
        seed=42
    )

PIPELINE_T_PATH = f"{CKPT}/pipeline_t"
if os.path.exists(PIPELINE_T_PATH):
    print("Memuat pipeline_t dari checkpoint...")
    pipeline_t = Pipeline.load(PIPELINE_T_PATH)
else:
    print("Membuat pipeline_t baru...")
    pipeline_t = Pipeline(stages=[assembler_t, gbt_t])
    pipeline_t.save(PIPELINE_T_PATH)
    print(f"pipeline_t disimpan di: {PIPELINE_T_PATH}")

PARAM_GRID_PATH = f"{CKPT}/param_grid/param_grid.pkl"
os.makedirs(f"{CKPT}/param_grid", exist_ok=True)
if os.path.exists(PARAM_GRID_PATH):
    print("Memuat param_grid dari checkpoint...")
    with open(PARAM_GRID_PATH, "rb") as f:
        param_grid = pickle.load(f)
else:
    print("Membuat param_grid baru...")
    param_grid = ParamGridBuilder() \
        .addGrid(gbt_t.maxDepth, [6, 8]) \
        .addGrid(gbt_t.maxIter,  [50, 100]) \
        .addGrid(gbt_t.stepSize, [0.05, 0.1]) \
        .build()
    with open(PARAM_GRID_PATH, "wb") as f:
        pickle.dump(param_grid, f)
    print(f"param_grid disimpan di: {PARAM_GRID_PATH}")

print(f"Total kombinasi parameter : {len(param_grid)}")
print(f"Total training jobs       : {len(param_grid) * 3} (3-fold CV)")
print(f"SPARKML_TEMP_DFS_PATH     : {os.environ['SPARKML_TEMP_DFS_PATH']}")

cross_val = CrossValidator(
    estimator=pipeline_t,
    estimatorParamMaps=param_grid,
    evaluator=eval_rmse,
    numFolds=3,
    parallelism=4,
    seed=42
)

print("\nMemulai CrossValidator... (estimasi 15-30 menit)")
cv_model = cross_val.fit(train_cv)
print("Tuning selesai!")

CV_MODEL_PATH = f"{VOLUME_PATH}/checkpoints/cv_model"
cv_model.write().overwrite().save(CV_MODEL_PATH)
print(f"CV model disimpan di: {CV_MODEL_PATH}")

import shutil
tmp_path = "/Volumes/workspace/default/nyc_taxi_volume/sparkml_tmp"
if os.path.exists(tmp_path):
    shutil.rmtree(tmp_path)
    print(f"Temp files dibersihkan.")

In [0]:
from pyspark.ml.tuning import CrossValidatorModel

loaded_cv_model = CrossValidatorModel.load(f"{VOLUME_PATH}/checkpoints/cv_model")

print("Akses Model Terbaik:")
print(loaded_cv_model.bestModel)

print("\nSkor RMSE rata-rata tiap kombinasi:")
print(loaded_cv_model.avgMetrics)

best_gbt_params = loaded_cv_model.bestModel.stages[-1]
print("\nParameter Optimal:")
print(f"Max Depth: {best_gbt_params.getOrDefault('maxDepth')}")
print(f"Max Iter: {best_gbt_params.getOrDefault('maxIter')}")
print(f"Step Size: {best_gbt_params.getOrDefault('stepSize')}")

In [0]:
assembler_t = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features",
    handleInvalid="skip"
)
gbt_t = GBTRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    maxBins=20,
    seed=42
)
pipeline_t = Pipeline(stages=[assembler_t, gbt_t])

param_grid = ParamGridBuilder() \
    .addGrid(gbt_t.maxDepth, [8, 10]) \
    .addGrid(gbt_t.maxIter,  [20, 50]) \
    .addGrid(gbt_t.stepSize, [0.05, 0.1]) \
    .build()

In [0]:
CV_MODEL_PATH = f"{VOLUME_PATH}/checkpoints/cv_model"
cv_model      = CrossValidatorModel.load(CV_MODEL_PATH)
print(f"CV model berhasil dimuat dari: {CV_MODEL_PATH}")

In [0]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import PipelineModel
import json

CKPT = f"{VOLUME_PATH}/checkpoints"

best_pipeline_model = cv_model.bestModel
best_gbt_stage      = best_pipeline_model.stages[-1]

best_depth  = best_gbt_stage.getMaxDepth()   # 8
best_iter   = best_gbt_stage.getMaxIter()    # 50
best_step   = best_gbt_stage.getStepSize()   # 0.1

print("=" * 60)
print("RETRAIN GBT dengan Best Hyperparameter di Data Penuh")
print("=" * 60)
print(f"  maxDepth  : {best_depth}")
print(f"  maxIter   : {best_iter}")
print(f"  stepSize  : {best_step}")
print(f"  Data      : {train_df.count():,} baris (train_df penuh)")

gbt_best = GBTRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    maxDepth=best_depth,
    maxIter=best_iter,
    stepSize=best_step,
    subsamplingRate=0.8,
    minInstancesPerNode=5,
    maxBins=32,
    seed=42
)

print("\nTraining GBT Best Params di data penuh...")
gbt_best_model = gbt_best.fit(train_df)
print("Training selesai!")

gbt_best_preds  = gbt_best_model.transform(test_df)
gbt_best_result = evaluate(gbt_best_preds, "GBT Best Params (Full Data)")

GBT_BEST_PATH = f"{VOLUME_PATH}/best_gbt_fulldata_model"
gbt_best_model.write().overwrite().save(GBT_BEST_PATH)
print(f"\nModel disimpan: {GBT_BEST_PATH}")

with open(f"{CKPT}/gbt_best_result.json", "w") as f:
    json.dump(gbt_best_result, f, indent=2)

In [0]:
from pyspark.ml.regression import GBTRegressionModel
import json

CKPT       = f"{VOLUME_PATH}/checkpoints"
GBT_BEST_PATH = f"{VOLUME_PATH}/best_gbt_fulldata_model"

gbt_best_model = GBTRegressionModel.load(GBT_BEST_PATH)
print(f"gbt_best_model loaded dari: {GBT_BEST_PATH}")
print(f"  maxDepth  : {gbt_best_model.getMaxDepth()}")
print(f"  maxIter   : {gbt_best_model.getMaxIter()}")
print(f"  stepSize  : {gbt_best_model.getStepSize()}")

with open(f"{CKPT}/gbt_best_result.json", "r") as f:
    gbt_best_result = json.load(f)
print(f"\ngbt_best_result loaded:")
print(f"  RMSE : {gbt_best_result['RMSE']:.4f}")
print(f"  MAE  : {gbt_best_result['MAE']:.4f}")
print(f"  R²   : {gbt_best_result['R2']:.4f}")

gbt_best_preds = gbt_best_model.transform(test_df)
print(f"\nPrediksi berhasil digenerate: {gbt_best_preds.count():,} baris")

In [0]:
import builtins

print("\n" + "=" * 65)
print("PERBANDINGAN FINAL — Semua di test_df (seed=42, 15%)")
print("=" * 65)
print(f"{'Model':<35} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print("-" * 65)

all_results = [lr_result, gbt_result, gbt_best_result]
for r in all_results:
    best_marker = " ← BEST" if r["RMSE"] == builtins.min(x["RMSE"] for x in all_results) else ""
    print(f"{r['model']:<35} {r['RMSE']:>8.4f} {r['MAE']:>8.4f} {r['R2']:>8.4f}{best_marker}")
    
print("\n" + "=" * 65)
print("GBT Default  vs  GBT Best Params (apple-to-apple)")
print("=" * 65)
print(f"{'Metrik':<10} {'GBT Default':>15} {'GBT Best':>15} {'Improvement':>15}")
print("-" * 65)
for metric in ["RMSE", "MAE", "R2"]:
    before = gbt_result[metric]
    after  = gbt_best_result[metric]
    if metric == "R2":
        improved = after > before
        diff     = after - before
        label    = "LEBIH BAIK ✓" if improved else "lebih buruk"
        sign     = "+" if diff > 0 else ""
    else:
        improved = after < before
        diff     = after - before
        label    = "LEBIH BAIK ✓" if improved else "lebih buruk"
        sign     = "+" if diff > 0 else ""
    print(f"{metric:<10} {before:>15.4f} {after:>15.4f} "
          f"{sign}{diff:>10.4f}   {label}")